# 2-clean&filter

In [25]:
# TODO: vérifier si tout OK pour fusion id_acteur/id_orateur/nom_orateur. notamment si ils correspondent bien au meme point
# TODO: vérif comparaison de id acteur =! id orateur

## 2.1 pré-nettoyage, pré-filtrage et pré-recodages

In [26]:
import pandas as pd
import re

# Charger le df concaténé des deux législatures
df = pd.read_csv(
    "../data/interim/extract_15_16_concat.csv",
    low_memory=False,
    dtype={
        "id_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
    },
)

print("Shape du df chargé : ", df.shape)

# ==============================
# Pré-nettoyage et pré-filtrage
# ==============================
"""
nb : précision choix 
- exclusion président.e :
role_debat n'est pas bien identifié, utiliser nom_orateur
(avant de le recoder/nettoyer car sinon risque perte par remplacement)
- id_acteur vs id_orateur :
certains cas (~3000) id_orateur plus précis (un code PA) que id_acteur qui a PA0
Mais en fait ce sont des 100% interruptions avec quasi toujours plusieurs locuteurs.
id_orateur en renvoie (mal) un seul -> on préfère garder le PA0 (neutre)
TODO : Rares exceptions avec Dupont-morretti seul, etc. -> mais bon…
"""

# Exclure les prises de parole de "Mme la présidente" et "M. le président"
df = df[~df["nom_orateur"].str.strip().isin(["M. le président", "Mme la présidente"])]

# Ne garder que le code style NORMAL
df = df[df["code_style"] == "NORMAL"]

# Changer les missing values pour non_précisé (majoritaire) dans Code_parole
df["code_parole"] = df["code_parole"].fillna("non_précisé")

# Garder une trace de la longueur des interventions brutes
df["len_texte_brut"] = df["texte"].str.len()

# Stabiliser le id_orateur pour être au format AN
df["id_orateur"] = "PA" + df["id_orateur"]
# Remplacer les valeurs manquantes de id_acteur par id_orateur quand disponible
df["id_acteur"] = df["id_acteur"].combine_first(df["id_orateur"])

# ===========================================================
# Récupérer et nettoyer les noms les plus fréquents
# pour chaque id_acteur sauf PA0 et les id_acteur manquants
# ===========================================================

# ========== Recoder par noms les plus fréquents ==========

# Nom le plus fréquent
most_frequent_name = df.groupby("id_acteur")["nom_orateur"].agg(
    lambda x: x.dropna().mode().iloc[0] if x.dropna().size > 0 else None
)  # version plus stable que value_counts().idxmax() en cas d'ex-aequo


# Renvoyer le nom le plus fréquent sauf si id_acteur == PA0 ou id_acteur est manquant
# Limite de la fonction : invisibilise les rares cas d'interventions
# mal identifiées par leur PA, mais qui ont le bon nom
# (ici le nom majoritaire sera renvoyé)
def get_most_frequent_name(row):
    """
    Récupération de la forme la plus fréquente du nom,
    uniquement pour acteurs différents de PA0.
    if PA0 : nom brut, else : nom le plus fréquent pour cet id.
    /!\ Limite de la fonction : invisibilise les rares cas d'interventions
    mal identifiées par leur PA, mais qui ont le bon nom (ici le nom majoritaire sera renvoyé)
    """
    if row["id_acteur"] == "PA0" or pd.isna(row["id_acteur"]):
        return row["nom_orateur"]
    return most_frequent_name.get(row["id_acteur"], row["nom_orateur"])


df["nom_orateur_clean"] = df.apply(get_most_frequent_name, axis=1)

# ========== Nettoyer les noms d'orateurs ==========


def nettoyer_nom(texte):
    if not isinstance(texte, str):
        return texte
    # Supprimer les balises HTML/XML
    texte = re.sub(r"<[^>]+>", "", texte)
    # Supprimer contenu entre parenthèses
    texte = re.sub(r"\([^)]*\)", "", texte)
    # Supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    # uniformise pour avoir les bons apostrophes
    texte = texte.replace("’", "'")
    return texte


df["nom_orateur_clean"] = df["nom_orateur_clean"].apply(nettoyer_nom)

print("Shape du df après pré-nettoyage et pré-filtrage : ", df.shape)


Shape du df chargé :  (1128128, 29)
Shape du df après pré-nettoyage et pré-filtrage :  (683680, 31)


## 2.2 Match des infos sur les députés (données datan)

### 2.2.1 Match des infos générales

In [27]:
# ==============================
# MATCH DONNÉES DÉPUTÉS
# ==============================

df_deputes = pd.read_csv("../data/raw/id-dep/deputes-historique(datan-datagouv).csv")

# suppression des colonnes non utiles qui introduisent soucis parsing
df_deputes = df_deputes.drop(
    columns=[
        "mail",
        "twitter",
        "facebook",
        "website",
        "active",
        "scoreParticipationSpecialite",
        "datePriseFonction",
        "groupe",
        "naissance",
    ]
)

# ======Fusion des données députés======

print("shape avant fusion:", df.shape)

assert df_deputes["id"].is_unique, "ids du df_deputes non uniques !"

# Merger et virer la col id pour éviter doublon
df = df.merge(
    df_deputes,
    left_on="id_acteur",
    right_on="id",
    how="left",
    suffixes=("", "_dep"),
    validate="many_to_one",  # check if merge keys are unique in right dataset
).drop(columns=["id"])  # supprimer la colonne id du df_deputes

print("shape après fusion données députés:", df.shape)

shape avant fusion: (683680, 31)
shape après fusion données députés: (683680, 48)


### 2.2.2 Match temporel des affiliations

In [28]:
# ======================================================
# AFFILIATION PARTISANE
# Logique suivie :
# 1. récupérer le groupe a date d'intervention si dispo
# 2. fallback sur dernière affiliation connue (groupe/groupeabrev)
# 3. variable supplémentaire avec gouvernement
# = écraser affil par GVT si qualite_orateur précise fonction gouvernementale
# ======================================================


# ======================================================
# RECODAGE ET MATCH TEMPOREL DES AFFILIATIONS PARTISANES
# cf. affiliation lors de telle prise de parole
# ======================================================


# ========== Recodage des dénominations de groupes ==========
"""
nb : ici choix de recoder avec les nom des partis,
car ils sont moins sensible aux évolutions marginales de noms,
même si en réalité les groupes parlementaires sont + larges que les partis
et peuvent servir à accueillir des NI d'étiquettes diverses
"""

# Lecture du fichier d'affiliation par périodes
df_affiliation = pd.read_csv(
    "../data/raw/id-dep/datan_affiliations.csv", encoding="latin1", sep=";"
)  # format dégueu

# Recodage des partis pour stabilité temporelle des noms
recodage_affiliation = {
    "RE": "REN",
    "EPR": "REN",
    "LAREM": "REN",
    "MODEM": "DEM",
    "SOC": "SOC-A",
    "NG": "SOC-A",
    "LFI-NUPES": "LFI",
    "FI": "LFI",
    "UDI-AGIR": "UDI",
    "UDI-A-I": "UDI",
    "LC": "UDI",
    "UDI_I": "UDI",
    "UDI-I": "UDI",
    "ECOLO": "ECO",
    "GDR-NUPES": "GDR",
    "LT": "LIOT",
    # Garde pour trace mais pas nécessaire car pas de changement
    # "LIOT": "LIOT",
    # "LR": "LR",
    # "RN": "RN",
    # "MODEM": "MODEM",
    # "LFI": "LFI",
    # "HOR": "HOR",
    # "DEM": "DEM",
}

# Application du recodage des noms de partis au df d'affiliation
df_affiliation["libelleAbrev"] = df_affiliation["libelleAbrev"].astype(str).str.strip()
df_affiliation["parti_recod"] = df_affiliation["libelleAbrev"].replace(
    recodage_affiliation
)

# ========== Match temporel des affiliations ==========

"""
nb : Plutôt qu'un merge foireux, parti sur un lookup ligne à ligne
(= pb des orateurs non députés qui étaient pas présents, etc.)
Le fichier est suffisamment réduit pour que le surplus de calcul soit pas un pb
nb : attention aux bornes temporelles (cf.normalize() pour ignorer l'heure)
"""

# préparation des dates
df["dateSeance_ts"] = pd.to_datetime(
    df["dateSeance"], format="%Y%m%d%H%M%S%f", errors="raise"
)
df_affiliation["dateDebut"] = pd.to_datetime(
    df_affiliation["dateDebut"], errors="raise"
)
df_affiliation["dateFin"] = pd.to_datetime(df_affiliation["dateFin"], errors="raise")
# aviser si jamais besoin un jour de traiter des affiliations en cours
# df_affiliation["dateFin"] = df_affiliation["dateFin"].fillna(pd.Timestamp("2100-01-01"))

# indexer par mpId pour lookup rapide
aff_by_mp = {
    mp: g[["dateDebut", "dateFin", "parti_recod"]].to_dict("records")
    for mp, g in df_affiliation.groupby("mpId")
}


# Fonction de recodage temporel des affiliations
def get_parti_for_row(row):
    """
    Retourne l'affiliation partisane recodée correspondant à la date de séance.

    La fonction :
    - lit `id_acteur` (assimilé à `mpId`) et `dateSeance_ts` sur la ligne ;
    - parcourt les périodes d'affiliation de ce député (si présent dans aff_by_mp);
    - renvoie `parti_recod` si `dateSeance_ts` (normalisée au jour) est comprise
    entre `dateDebut` et `dateFin` (bornes incluses).
    """
    mp = row.get("id_acteur")  # correspond au mpId
    # gérer le cas des orateurs non députés ou autre type intervention
    if pd.isna(mp) or mp not in aff_by_mp:
        return None
    # récupérer le ts de l'intervention
    ts = row.get("dateSeance_ts")
    if pd.isna(ts):
        return None
    # retourner l'affiliation qui colle à la date d'intervention
    for rec in aff_by_mp[mp]:
        # attention : .normalize() pour ignorer l'heure car sinon hors des bornes de fin
        if rec["dateDebut"] <= ts.normalize() <= rec["dateFin"]:
            return rec["parti_recod"]
    return None


# application du match temporel
df["affiliation_mandat_députés"] = df.apply(get_parti_for_row, axis=1)

# Pas parfait mais pour avoir une idée :
print(
    "affectés :",
    df["affiliation_mandat_députés"].notna().sum(),
    # Eux on sait pas (pas députés, autre code parole intervention, etc.)
    "| non affectés :",
    df["affiliation_mandat_députés"].isna().sum(),
)


affectés : 563127 | non affectés : 120553


### 2.2.3 Fallback des affiliations manquantes

#### Forcer le renvoi d'une affiliation si groupeAbrev connu

In [29]:
# ============================================================
# GESTION AFFILIATIONS MANQUANTES ET MEMBRES DU GOUVERNEMENT
# - Fallback pour les affiliations manquantes
# - Gestion des cas limites (RN, etc.)
# - Création catégorie "GOUV" pour les membres du gouvernement

# ============================================================

# ========= Fallback affiliation manquantes par groupeAbrev ==========

# TODO: aviser si va à Matthias + ce que veux faire de EDS / AGIR-E
# TODO: veut aussi dire qu'on bourre des cas limites, genre un vieux député UMP qui revient au gouv etc.
# Aussi possible de gérer à la main les cas UMP, etc.

# Forcer une affiliation avec le groupe "groupeAbrev" du fichier info députés
df["affiliation"] = df["affiliation_mandat_députés"].combine_first(df["groupeAbrev"])
# réutiliser le même recodage que pour les affiliations
df["affiliation"] = df["affiliation"].replace(recodage_affiliation)
# Et gérer les nouvelles dénominations propres groupeAbrev
df["affiliation"] = df["affiliation"].replace({"LES-REP": "LR", "UMP": "LR"})

# Diagnostic des cas concernés par le fallback groupeAbrev
# = c'est surtout des membres du gouv qui avaient pas d'affiliation de mandat députés
# + quelques rares cas d'interv députés ou on manque parfois l'affiliation dynamique (bornes ?)

mask_fallback = df["affiliation_mandat_députés"].isna() & df["affiliation"].notna()

print("=== Cas concernés par le fallback via groupeAbrev ===")
print("Nombre d'interventions concernées :", mask_fallback.sum())
print(
    "Nombre d'id_acteur uniques :",
    df.loc[mask_fallback, "id_acteur"].nunique(dropna=True),
)

print("\nListe des orateurs concernés :")
print(df.loc[mask_fallback, "nom_orateur_clean"].dropna().unique())


=== Cas concernés par le fallback via groupeAbrev ===
Nombre d'interventions concernées : 64950
Nombre d'id_acteur uniques : 60

Liste des orateurs concernés :
['M. Bruno Le Maire' 'M. Édouard Philippe' 'M. Olivier Dussopt'
 'M. Benjamin Griveaux' 'M. Stéphane Travert' 'Mme Brune Poirson'
 'M. Christophe Castaner' 'M. Jean-Yves Le Drian' 'Mme Bérangère Abba'
 'Mme Barbara Pompili' 'Mme Élisabeth Borne' 'M. Jean-Baptiste Djebbari'
 'Mme Nathalie Elimas' 'Mme Brigitte Bourguignon' 'Mme Brigitte Klinkert'
 'Mme Annick Girardin' 'M. Mounir Mahjoubi' 'M. Gérald Darmanin'
 'M. Olivier Véran' 'Mme Agnès Pannier-Runacher' 'M. Marc Fesneau'
 'M. Clément Beaune' 'Mme Sarah El Haïry' 'Mme Geneviève Darrieussecq'
 'M. Franck Riester' 'Mme Olivia Grégoire' 'M. Laurent Pietraszewski'
 'M. François de Rugy' 'Mme Amélie de Montchalin' 'Mme Nadia Hai'
 'Mme Roselyne Bachelot' 'Mme Christelle Dubos' 'M. Gabriel Attal'
 'M. Adrien Taquet' 'M. Joël Giraud' 'Mme Valérie Boyer'
 'Mme Prisca Thevenot' 'Mme C

#### Forcer affiliation des RN qui étaient en NI (étaient pas assez pour groupe)

In [30]:
# ========= Gestion cas limites RN ==========

# Recodage des RN de la XVe législature au bloc RN
# nb = choix = initialement en NI car pas assez nombreux pour former un groupe

liste_NI_RN = [
    "PA720822",  # Bruno Bilde
    "PA720668",  # Sébastien Chenu
    "PA720468",  # Emmanuel Blairy
    "PA720614",  # Marine Le Pen
    "PA719436",  # Nicolas Meizonnet
    "PA720802",  # Catherine Pujol
    "PA719608",  # Emmanuelle Ménard, rattachée au RN entre 2017 et 2022 mais plus entre 2022 et 2024
    "PA720606",  # Ludovic Pajot
    "PA606212",  # Gilbert Collard
    "PA720798",  # Louis Aliot
]

# Date seuil : fin de la 15e législature
date_seuil = pd.Timestamp("2022-06-21")

# Condition combinée :
condition_NI_RN = (df["id_acteur"].isin(liste_NI_RN)) & (
    df["dateSeance_ts"].dt.normalize() < date_seuil
)  # dt.normalize() pour ignorer l'heure et éviter soucis de bornes


# Application de la modalité uniquement pour les lignes correspondant à la condition
df.loc[condition_NI_RN, "affiliation"] = "RN"

# Vérification
print("Lignes recodées RN :", condition_NI_RN.sum())
print(
    "Affiliation recodées pour",
    df.loc[condition_NI_RN, "id_acteur"].nunique(),
    "id_acteur uniques",
)

# vérification des cas sans affiliation :
print("Ceci ne modifie pas nombre sans affiliation : simple recodage NI vers RN")


Lignes recodées RN : 6701
Affiliation recodées pour 10 id_acteur uniques
Ceci ne modifie pas nombre sans affiliation : simple recodage NI vers RN


### Création d'une variable sur-imprimant l'appartenance au gouv

In [31]:
# ========== Création variable avec GOUV ==========

# renvoyer les membres du gouv à une catégorie "GOUV" pour les différencier
"""
nb : traçabilité
/!\ ici on veut récup membres du gouv, souvent en sans affiliation
mais on veut aussi forcer leur etiquette gvt même quand ils ont une affiliation de député
(ex : ministre qui est aussi député)

Logique de recodage :
Recoder membres GVT, uniquement si != PA0 (= garder cohérence avec cas précédents)
si une des conditions suivantes est vérifiée,
- ministre -> ok, 96 personnes pour 130 qualité, mais exclure le cas de Justin Trudeau et 19 cas PA0
- garde des sceaux (pas toujours co-qualifié de ministre) : ok, 2 bien Dupond-Moretti / Belloubet (même si 10 PA0)
- secrétaire d’État -> 40 personnes pour 53 qualité correspondantes, OK (2 PA0)
= basé sur la lecture des résultats de :
df["qualite_orateur"].value_counts()

-> mais il faut exclure "Premier ministre du Canada" -> 2 occurences 
Autre option : exclure des PA PA-107309 = Justin Trudeau, Premier ministre du Canada
"""

# masque condition membres gouvernement
mask_gvt = (
    df["qualite_orateur"].str.contains(
        "ministre|garde des sceaux|secrétaire d[’']État",
        case=False,
        na=False,
        regex=True,
    )
    & (df["id_acteur"] != "PA0")
    & (df["id_acteur"] != "PA-107309")
)  # exclure Justin Trudeau, "Premier ministre du Canada"

# ========== Création nouvelle variable avec GVT ==========
df["affiliation_et_gouv"] = df["affiliation"]  # conserver l'affiliation initiale
df.loc[mask_gvt, "affiliation_et_gouv"] = "GOUV"

# Vérification des cas affectés recodage GOUV
print("Lignes recodées GOUV :", mask_gvt.sum())
print(
    "Affiliation recodées pour",
    df.loc[mask_gvt, "id_acteur"].nunique(),
    "id_acteur uniques",
)

Lignes recodées GOUV : 110358
Affiliation recodées pour 109 id_acteur uniques


# GESTION DES CAS RESTANTS

In [32]:
# ======================
# TODO: AFFILIATIONS :
# ======================
# TODO : explorer les affiliation manquantes pour identifier les cas limites


In [33]:
# vérification des cas sans affiliation :
print(
    "Nombre restant d'interventions sans affiliation :",
    df["affiliation"].isna().sum(),
)
print(
    "Nombre restant d'id_acteur uniques sans affiliation :",
    df[df["affiliation"].isna()]["id_acteur"].nunique(),
)

print(
    "Nombre restant d'interventions sans affiliation_et_gouv :",
    df["affiliation_et_gouv"].isna().sum(),
)
print(
    "Nombre restant d'id_acteur uniques sans affiliation_et_gouv :",
    df[df["affiliation_et_gouv"].isna()]["id_acteur"].nunique(),
)

Nombre restant d'interventions sans affiliation : 55603
Nombre restant d'id_acteur uniques sans affiliation : 174
Nombre restant d'interventions sans affiliation_et_gouv : 12014
Nombre restant d'id_acteur uniques sans affiliation_et_gouv : 131


In [34]:
restant = df[(df["affiliation_et_gouv"].isna()) & (df["id_acteur"] != "PA0")]
restant.head()

,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,nombreMandats,experienceDepute,scoreParticipation,scoreLoyaute,scoreMajorite,dateMaj,dateSeance_ts,affiliation_mandat_députés,affiliation,affiliation_et_gouv
378,CRSANR5L15S2019O1N178,NaN,NaN,20190314150000000,jeudi 14 mars 2019,2,178,AN,15,Session ordinaire 2018-2019,...,NaN,NaN,NaN,NaN,NaN,NaN,2019-03-14 15:00:00,None,NaN,NaN
2577,CRSANR5L15S2020O1N119,NaN,NaN,20200109090000000,jeudi 09 janvier 2020,Unique,119,AN,15,session ordinaire 2019-2020,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-09 09:00:00,None,NaN,NaN
2578,CRSANR5L15S2020O1N119,NaN,NaN,20200109090000000,jeudi 09 janvier 2020,Unique,119,AN,15,session ordinaire 2019-2020,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-09 09:00:00,None,NaN,NaN
2579,CRSANR5L15S2020O1N119,NaN,NaN,20200109090000000,jeudi 09 janvier 2020,Unique,119,AN,15,session ordinaire 2019-2020,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-09 09:00:00,None,NaN,NaN
2581,CRSANR5L15S2020O1N119,NaN,NaN,20200109090000000,jeudi 09 janvier 2020,Unique,119,AN,15,session ordinaire 2019-2020,...,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-09 09:00:00,None,NaN,NaN


In [35]:
# les gens qui ne sont ni PA0 ni affiliés à un parti ou au gouv
# = sans doute des extérieurs ???

df[(df["affiliation_et_gouv"].isna()) & (df["id_acteur"] != "PA0")][
    ["nom_orateur_clean"]
].value_counts()


nom_orateur_clean      
M. Jean-Paul Delevoye      79
M. Pierre Moscovici        59
Mme Olivia Grégoire        37
M. Lyes Louffok            26
M. Erwan Lecœur            21
                           ..
M. Roland Courteau          1
M. Gérard Larcher           1
M. François-Noël Buffet     1
M. Emmanuel Capus           1
Mme Éliane Assassi          1
Name: count, Length: 119, dtype: int64

In [36]:
# Cas où un même id_acteur a plusieurs valeurs différentes de affiliation_et_gouv
tmp = df[["id_acteur", "nom_orateur_clean", "affiliation_et_gouv"]].copy()
tmp["affiliation_et_gouv_norm"] = tmp["affiliation_et_gouv"].fillna("<<NA>>")

# ids avec au moins 2 modalités différentes (en comptant NA)
ids_multi_affil = (
    tmp.groupby("id_acteur")["affiliation_et_gouv_norm"]
    .nunique()
    .loc[lambda s: s > 1]
    .index
)

cas_diff = tmp[tmp["id_acteur"].isin(ids_multi_affil)].copy()

print("Nombre d'id_acteur avec plusieurs valeurs de affiliation_et_gouv :", len(ids_multi_affil))
print("Nombre total de lignes concernées :", len(cas_diff))

display(
    cas_diff.groupby(["id_acteur", "nom_orateur_clean"])["affiliation_et_gouv_norm"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="valeurs_affiliation_et_gouv")
    .sort_values(["nom_orateur_clean", "id_acteur"])
)

Nombre d'id_acteur avec plusieurs valeurs de affiliation_et_gouv : 162
Nombre total de lignes concernées : 176008


,id_acteur,nom_orateur_clean,valeurs_affiliation_et_gouv
93,PA720422,M. Adrien Quatennens,"[LFI, NI]"
137,PA722086,M. Adrien Taquet,"[GOUV, NI, REN]"
156,PA794914,M. Alexandre Vincendet,"[HOR, LR]"
27,PA421348,M. André Villiers,"[HOR, UDI]"
8,PA267355,M. Antoine Herth,"[AGIR-E, UDI]"
...,...,...,...
22,PA336175,Mme Sylvia Pinel,"[LIOT, NI]"
95,PA720500,Mme Valérie Petit,"[AGIR-E, REN]"
63,PA719194,Mme Yolaine de Courson,"[DEM, EDS, NI, REN]"
50,PA717161,Mme Élisabeth Borne,"[GOUV, REN]"


In [ ]:
# TODO: comparer ce que ça donne pour gouv entre affiliation_mandat_députés et affiliation_et_gouv pour voir si ça correspond bien
# = cf on en trouvait sans doute comme ça des vides dans affiliation_mandat_députés qui étaient en fait des membres du gouv
# et que là on recode selon leur affiliation et pas en GOUV si l'info qualité orateur est pas bonne
# pas identifié ministre machin car info manquante, autre statut comme rapporteur, etc.

# TODO: géréer les cas limites gouv quand sont rapporteurs, etc. (darmanin, EDM, etc.)
# TODO: sans doute donc gérer chaque type qui a GVT et autre affilaition pour voir si normal ?

# Gestion des cas limites

In [ ]:
# ==============================
# vérif et possibles soucis :
# ==============================

# TODO: matthias : check les cas particuliers.
# TODO : léo, voir ces machins avec matthias ensuite pour clarifier.
# Et voir pourquoi passé par un isin plutôt que ==

# TODO : pour gestion possible des cas limites (traitement post affiliation RN vs gauche)
# possible de renvoyer à la toute fin une liste des gens qui ont plusieurs affiliation dans le temps
# (y compris sans étiquette) pour pouvoir gérer au cas par cas le choix de renvoi de l'affiliation.

# TODO : Comprendre pq Bruneel = ["PA720546"] est laissé en Valeur manquante sur une intervention le 9 janvier 2023 et Boyer = ["PA720546"] sur intervention du 7 novembre 2020
# -> RETOUR LM : c'est ok une fois qu'on force les affil.

In [ ]:
# ========= Diagnostic des cas concernés par le fallback ==========

# Cas concernés : affiliation mandat manquante, mais groupeAbrev disponible
mask_fallback = df["affiliation_mandat_députés"].isna() & df["groupeAbrev"].notna()

fallback_cases = df.loc[
    mask_fallback,
    [
        "id_acteur",
        "nom_orateur_clean",
        "qualite_orateur",
        "groupeAbrev",
        "dateSeance_ts",
    ],
].copy()

# Reconstituer l'affiliation qui serait attribuée par le fallback
fallback_cases["affiliation_fallback"] = fallback_cases["groupeAbrev"]
fallback_cases["affiliation_fallback"] = fallback_cases["affiliation_fallback"].replace(
    recodage_affiliation
)
fallback_cases["affiliation_fallback"] = fallback_cases["affiliation_fallback"].replace(
    {"LES-REP": "LR", "UMP": "LR"}
)

# Print des infos
print("=== Cas concernés par le fallback via groupeAbrev ===")
print("Nombre d'interventions concernées :", len(fallback_cases))
print("Nombre d'id_acteur uniques :", fallback_cases["id_acteur"].nunique(dropna=True))
print(
    "Nombre d'orateurs uniques :",
    fallback_cases["nom_orateur_clean"].nunique(dropna=True),
)

print("\nListe des orateurs concernés :")
print(fallback_cases["nom_orateur_clean"].dropna().unique())

display(
    fallback_cases[
        [
            "id_acteur",
            "nom_orateur_clean",
            "groupeAbrev",
            "affiliation_fallback",
        ]
    ]
    .value_counts()
    .reset_index(name="n_interventions")
)

In [ ]:
# ========= Cas limites GOUV =============

# vérification des cas sans affiliation :
print(
    "Nombre d'id_acteur uniques sans affiliation :",
    df[df["affiliation_mandat_députés"].isna()]["id_acteur"].nunique(),
)
print("\nValeur counts des id_acteur sans affiliation (top):")
print(
    df[df["affiliation_mandat_députés"].isna()][["id_acteur", "nom_orateur_clean"]]
    .value_counts()
    .head()
)

# Vérifier les cas où affiliation n'est pas nulle mais avec qualité orateur spécifique
# = membres du gouv mais qui sont députés et flaguent donc avec une affiliation députés

mask_affil_with_qualite = df["affiliation_mandat_députés"].notna() & df[
    "qualite_orateur"
].str.contains("ministre|garde des sceaux|secrétaire d'État", case=False, na=False)

print(
    "Nombre de lignes avec affiliation ET qualité gouvernementale :",
    mask_affil_with_qualite.sum(),
)
print("\nAffiliations pour ces cas :")
print(
    df.loc[mask_affil_with_qualite, "affiliation_mandat_députés"].value_counts().head()
)

print("\nExemples de ces lignes :")
print(
    df.loc[
        mask_affil_with_qualite,
        ["nom_orateur_clean", "qualite_orateur", "affiliation_mandat_députés"],
    ]
    .drop_duplicates()
    .head()
)


In [ ]:
df.loc[
    mask_affil_with_qualite,
    ["nom_orateur_clean", "id_acteur", "affiliation_mandat_députés"],
].value_counts()

In [ ]:
# vérification des cas sans affiliation :
print(
    "Nombre restant d'interventions sans affiliation :",
    df["affiliation"].isna().sum(),
)
print(
    "Nombre restant d'id_acteur uniques sans affiliation :",
    df[df["affiliation"].isna()]["id_acteur"].nunique(),
)
print("\nValue counts des id_acteur sans affiliation (top):")
print(
    df[df["affiliation"].isna()][["id_acteur", "nom_orateur_clean"]]
    .value_counts()
    .head()
)


In [ ]:
missing_affil = df[df["affiliation"].isna()][
    ["id_acteur", "nom_orateur_clean", "qualite_orateur"]
]
missing_affil = missing_affil[missing_affil["id_acteur"] != "PA0"]
missing_affil

missing_affil["nom_orateur_clean"].value_counts()

missing_affil["nom_orateur_clean"].value_counts()

### Trucs Matthias
Mais possiblement ok depuis qu'on force l'allifilation ? 

In [28]:
# TODO : léo, voir ces machins avec matthias ensuite pour clarifier.
# Et voir pourquoi passé par un isin plutôt que ==

In [29]:
# # solution temporaire sur 2 cas étranges
# Boyer = ["PA330684"]  # cas similaire sur intervention du 7 novembre 2020

# df.loc[df["id_acteur"].isin(Boyer), "groupe&gvt_affiliation"] = "LR"

# Bruneel = [
#     "PA720546"
# ]  # ici cas étrange sur une intervention le 9 janvier 2023, il a été laissé en valeur manquante alors que GDR

# df.loc[df["id_acteur"].isin(Bruneel), "groupe&gvt_affiliation"] = "GDR"

In [30]:
# # Reste des cas particuliers à replacer dans leur affiliation au moment de leurs fonctions gouvernementales respectives
# Bachelot = ["PA332"]

# df.loc[df["id_acteur"].isin(Bachelot), "groupe_all_affiliation"] = (
#     "NI"  # NI ou mettre valeur manquante ? pareil pour Philippe, Le Drian, Rousseau
# )

# Vautrin = ["PA267797"]

# df.loc[df["id_acteur"].isin(Vautrin), "groupe_all_affiliation"] = "REN"

# Philippe = ["PA345619"]

# df.loc[df["id_acteur"].isin(Philippe), "groupe_all_affiliation"] = "NI"

# Ledrian = ["PA1872"]

# df.loc[df["id_acteur"].isin(Ledrian), "groupe_all_affiliation"] = "NI"

# Rousseau = ["PA826635"]

# df.loc[df["id_acteur"].isin(Rousseau), "groupe_all_affiliation"] = "NI"

## Export

In [31]:
# Export du csv nettoyé
df.to_csv("../data/interim/data_cleaning_full.csv", index=False)

# # NB: certaines col du df_deputes introduisent une erreur à l'import/export
# # Elles ne sont pas utilisées ici, mais si besoin de les utiliser
# # forcer le QUOTE_ALL permet de résoudre
# # (cf : adresses et réseaux sociaux contenant saut de lignes = erreurs de parsing (cas eric.martineau))

KeyboardInterrupt: 

# PROVISOIRE !! Regrouper les interventions interrompues
ÇA NE MARCHE PAS POUR L'INSTANT !!!!!!


In [ ]:
# TODO: À affiner et vérifier la fusion interventions interrompues

# AVISER : pas le cas ici, mais envisager possible gestion des cas NaN
df_interruption = df[df["code_grammaire"].str.contains("INTERRUPTION")]
df_intervention = df[~df["code_grammaire"].str.contains("INTERRUPTION")]
# Si il fallait s'en assurer :
# is_interruption = df["code_grammaire"].str.contains("INTERRUPTION", na=False)
# df_interruption = df[is_interruption]
# df_intervention = df[~is_interruption]

# assert len(df_interruption) + len(df_intervention) == len(df), (
#     f"Lignes perdues lors du split ! "
#     f"{len(df)} ≠ {len(df_interruption)} + {len(df_intervention)} "
#     f"(NaN dans code_grammaire : {df['code_grammaire'].isna().sum()})"
# )

# TODO : NON ÇA VA PAS ÇA REGROUPE NAWAK ????

# ordinal_prise semble plus précis au niveau des intervenants
# = est constant quand interrompu là où les ordres obsolu et ptsodj changent
group_keys = ["uid", "dateSeance_ts", "id_acteur", "ordinal_prise"]

# agréger : concat texte, sommer longueur, garder premières infos utiles
agg = {
    "texte": lambda s: " ".join(s.dropna().astype(str)).strip(),
    "len_texte_brut": "sum",
    "code_parole": lambda s: ", ".join(sorted(set(s.dropna().astype(str)))),
    "id_syceron": lambda s: s.dropna().unique().tolist(),
    # "ordre_absolu_seance": "first", # list pour garder l'ordre des prises ?
    # "nom_orateur": "first",
    # "qualite_orateur": "first",
    # "id_orateur": "first",
    # "stime": "first",
}

# ajouter 'first' pour toutes les autres colonnes non clés/non déjà agrégées
for c in df_intervention.columns:
    if c not in group_keys and c not in agg:
        agg[c] = "first"

# Regroupe les interventions par clés communes et agrège les colonnes définies dans `agg`
df_intervention_grouped = (
    df_intervention.groupby(group_keys, dropna=False).agg(agg).reset_index()
)

# Recolle les interventions regroupées avec les interruptions
# puis aligne les colonnes sur le format d'origine
df_concat = pd.concat([df_intervention_grouped, df_interruption], ignore_index=True)[
    df_interruption.columns
]

# Retrier dans l'ordre chronologique et d'affichage de la séance
# nb : ici ok car gardé seulement first pour ordre_absolu_seance
# mais modif si jamais on avait gardé la liste complète des ordres
df_concat = df_concat.sort_values(
    by=["dateSeance_ts", "valeur_ptsodj", "ordre_absolu_seance"]
).reset_index(drop=True)

print(
    f"Regroupement des interventions interrompues \n"
    f"avant: {len(df)} | après: {len(df_concat)} "
    f"(interventions: de {len(df_intervention)} → à {len(df_intervention_grouped)}, "
    f"interruptions: {len(df_interruption)})"
)

# TODO : len_texte_brut sera à réajouter pour interv groupée
# , car là on à la trace de la longueur des interventions avant regroupement
# donc voir pour une fois regroupé ?
# Surtout, renvoyer l'info de longueur dans df regroupé peut porter à confusion ?
# ATTEND : J'EN FAIS DÉJÀ UNE SOMME NON :     "len_texte_brut": "sum",
# aviser, quitte a préciser que c'est une nouvelle col.

In [ ]:
# Export du csv concat nettoyé
df_concat.to_csv("../data/interim/data_cleaning_grouped.csv", index=False)

# # NB: certaines col du df_deputes introduisent une erreur à l'import/export
# # Elles ne sont pas utilisées ici, mais si besoin de les utiliser
# # forcer le QUOTE_ALL permet de résoudre
# # (adresses et réseaux sociaux contenant saut de lignes = erreurs de parsing (cas eric.martineau))

# EXPLORATION

In [ ]:
# TODO : aller voir parce que ça regroupe quand meme des trucs qui
# ont pas le même code parole
# donc voir le pourquoi du comment
# MAIS ON S'EN COGNE UN PEU SUR LE PRINCIPE ?
# ENFIN AVISER QUE JUSTE LES AVIS GOUV SOIT PAS REGROUPÉS
# AVEC UNE PRISE PAROLE PLUS LARGE ?
# CHANGE RIEN DE DRAMATIQUE SANS DOUTE.

df_concat["code_parole"].value_counts()[:-10]

code_parole
non_précisé                                 287786
PAROLE_1_2                                   92122
PAROLE_1_2, non_précisé                      15040
AVIS_COM_1_20                                10824
AVIS_GVT_1_20                                10097
AVIS_GVT_1_20, PAROLE_1_2                     3371
AVIS_COM_1_20, non_précisé                    2265
AVIS_COM_1_20, PAROLE_1_2                     1787
AVIS_GVT_1_20, non_précisé                    1096
AVIS_COM_1_20, PAROLE_1_2, non_précisé         684
AVIS_GVT_1_20, PAROLE_1_2, non_précisé         453
AVIS_COM_1_20, AVIS_GVT_1_20                    14
AVIS_COM_1_20, AVIS_GVT_1_20, PAROLE_1_2         5
Name: count, dtype: int64

In [ ]:
# ÇA REGROUPE NAWAK !!!!!

In [ ]:
chelou = df_concat[df_concat["code_parole"] == "AVIS_GVT_1_20, PAROLE_1_2"]
chelou.head

,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,nombreMandats,experienceDepute,scoreParticipation,scoreLoyaute,scoreMajorite,dateMaj,dateSeance_ts,groupe_députés_affiliation,groupe&gvt_affiliation,groupe_all_affiliation
430,CRSANR5L15S2017E1N003,None,None,20170706093000000,jeudi 06 juillet 2017,1,3,AN,15,Première session extraordinaire 2017,...,NaN,None,NaN,NaN,NaN,None,2017-07-06 09:30:00,None,GVT,None
467,CRSANR5L15S2017E1N004,None,None,20170706150000000,jeudi 06 juillet 2017,2,4,AN,15,Première session extraordinaire 2017,...,NaN,None,NaN,NaN,NaN,None,2017-07-06 15:00:00,None,GVT,None
817,CRSANR5L15S2017E1N006,None,None,20170710213000000,lundi 10 juillet 2017,2,6,AN,15,Première session extraordinaire 2017,...,NaN,None,NaN,NaN,NaN,None,2017-07-10 21:30:00,None,GVT,None
897,CRSANR5L15S2017E1N007,None,None,20170711150000000,mardi 11 juillet 2017,1,7,AN,15,Première session extraordinaire 2017,...,NaN,None,NaN,NaN,NaN,None,2017-07-11 15:00:00,None,GVT,None
1068,CRSANR5L15S2017E1N008,None,None,20170711213000000,mardi 11 juillet 2017,2,8,AN,15,Première session extraordinaire 2017,...,NaN,None,NaN,NaN,NaN,None,2017-07-11 21:30:00,None,GVT,None


Je comprends bien l’intention exposée par M. Lagarde. Je rappelle toutefois l’existence de la disposition dont vient de parler M. le rapporteur.De surcroît, le contrôle des assemblées a été sensiblement renforcé lors de la quatrième prorogation de l’état d’urgence, en juillet 2016. Dès le 25 juillet 2016, les commissions des lois des deux assemblées se sont ainsi vu transmettre copie des mesures prises sur le fondement de la loi du 3 avril 1955, ce qui a permis aux rapporteurs concernés de disposer d’une connaissance exhaustive de toutes ces mesures. J’en ai parlé abondamment au Sénat, hier, avec M. le rapporteur Michel Mercier. Les rapporteurs des deux commissions des lois sont informés de tout ce qui se passe pendant l’état d’urgence. L’avis du Gouvernement est donc défavorable. Vous me permettrez d’exprimer mon accord avec M. Larrivé : les moyens de contrôle dont disposent aujourd’hui les commissions des lois de l’Assemblée nationale et du Sénat sont très importants. Les informations les plus confidentielles sont communiquées à leurs présidents et rapporteurs respectifs. Vous comprendrez aisément que, pendant la Guerre de Quatorze, le comité parlementaire était informé des grandes lignes stratégiques, pas forcément de la tactique déployée sur le terrain. L’avis du Gouvernement reste donc défavorable.

Je comprends bien l’intention exposée par M. Lagarde. Je rappelle toutefois l’existence de la disposition dont vient de parler M. le rapporteur.De surcroît, le contrôle des assemblées a été sensiblement renforcé lors de la quatrième prorogation de l’état d’urgence, en juillet 2016. Dès le 25 juillet 2016, les commissions des lois des deux assemblées se sont ainsi vu transmettre copie des mesures prises sur le fondement de la loi du 3 avril 1955, ce qui a permis aux rapporteurs concernés de disposer d’une connaissance exhaustive de toutes ces mesures. J’en ai parlé abondamment au Sénat, hier, avec M. le rapporteur Michel Mercier. Les rapporteurs des deux commissions des lois sont informés de tout ce qui se passe pendant l’état d’urgence. L’avis du Gouvernement est donc défavorable.

In [ ]:
df[df["id_syceron"] == 983326][["nom_orateur_clean", "texte", "len_texte_brut"]]

,nom_orateur_clean,texte,len_texte_brut
78246,M. Gérard Collomb,Je comprends bien l’intention exposée par M. L...,791.0


In [ ]:
df[df["id_syceron"] == 983358][["nom_orateur_clean", "texte"]]

,nom_orateur_clean,texte
78261,M. Gérard Collomb,Vous me permettrez d’exprimer mon accord avec ...


In [ ]:
# Vérifier si code_parole varie au sein d'un même groupe
check = df_intervention.groupby(group_keys, dropna=False)["code_parole"].nunique()
print("Groupes avec code_parole non constant :", (check > 1).sum())

Groupes avec code_parole non constant : 24724
